In [ ]:
'''
繁蝉随堂笔记
'''

# 牛顿法与优化进阶

之前：
1. 用多项式去模仿超越函数
2. 用切线去不断逼近方程的根 牛顿法

优化的牛顿法：对导数再做一次牛顿求根


计算机 加法器  乘法：移位累加  减法：补码加法 除法：加减交替，不恢复余数法

$\mathrm{Min}\, f \to f' = 0$

牛顿-拉佛森方法

彗星轨道 月亮背面着陆  都在解决根的问题

康托洛维奇：起步离根不太远，导数不太病态的时候，牛顿法是局部二阶收敛，误差按照平方缩小。
$1e-3$,$1e-6$

路德维希 奥托 黑塞(19世纪 德国)：曲率表Hessian 海森矩阵
一维二阶信息 $f''(x)$
多维 每个方向$\frac{\partial^2 f}{\partial x_i^2}$ 每两个方向交叉$\frac{\partial^2 f}{\partial x_i \partial x_j}$ 都有一个二阶偏导
Hessian 海森矩阵  “地形曲率表”
$$
\mathbf{H} = \nabla^2 f =
\begin{bmatrix}
\dfrac{\partial^2 f}{\partial x_1^2} &
\dfrac{\partial^2 f}{\partial x_1 \partial x_2} &
\cdots &
\dfrac{\partial^2 f}{\partial x_1 \partial x_n} \\[8pt]
\dfrac{\partial^2 f}{\partial x_2 \partial x_1} &
\dfrac{\partial^2 f}{\partial x_2^2} &
\cdots &
\dfrac{\partial^2 f}{\partial x_2 \partial x_n} \\[8pt]
\vdots & \vdots & \ddots & \vdots \\[8pt]
\dfrac{\partial^2 f}{\partial x_n \partial x_1} &
\dfrac{\partial^2 f}{\partial x_n \partial x_2} &
\cdots &
\dfrac{\partial^2 f}{\partial x_n^2}
\end{bmatrix}
$$



拟牛顿法
sklean 逻辑回归 solver=‘L-BFGS’

Adam 2014  DL 一阶  梯度+动量+按坐标自适应步长

牛顿负责“看见曲率一步到位”，
拟牛顿负责“用梯度差猜曲率，省下 Hessian”，
Adam 负责“用梯度矩估计在随机噪声里自适应地走”。

LOSS 函数

求根是解$f=0$,求最优是解$f'=0$

1. 学习率
2. 病态 固定步长 “之”字形
3. 衡量

海森矩阵 8tb 求逆 $O(n^3)$  阻尼

离最优点很远，Hessian不正定、鞍点附近

一维：
$x<-x-\frac{f'(x)}{f(x)}$
多维：
$\mathbf{x} \leftarrow \mathbf{x} - \mathbf{H}^{-1} \nabla f(\mathbf{x})$

一维从哪里来？
$f(x)=(x-3)^2+1$
$f'(x)=2(x-3)$
$f''(x)= 2$
$x0=0 \to x1= 0-2(0-3)/2=3$

\begin{aligned}
&\text{二维病态碗：} f(x,y) = x^2 + 10y^2 \\
&\text{梯度} = (2x, 20y), \quad H = \text{diag}(2, 20) \\
&\text{牛顿法：一步收敛至 } (0,0) \\
&\text{梯度下降：当 } \eta = 0.08 \text{ 时，} y \text{ 方向每步乘以 } (1 - 20\eta) = 1 - 1.6 = -0.6, \text{ 导致变号震荡}
\end{aligned}


逻辑回归、广义线性模型：sklearn常用L-BFDS 或者newton-cg

scipy.optimize.mimize(method='BFGS')

L-BFGS-B 把这个函数最小化交给库，拟合、校准、反演

视觉与SLAM里面的Bundle.Adjustmengt、相机标定：高斯牛顿/Levernberg-Marquardt,牛顿法在最小二乘上的表亲；

PyTorch 里面有torch.optim.LBFGS，适合参数不多、loss光滑的小问题，不太适合训练大模型。


XGBoost（Extreme Gradient Boosting）极端梯度提升
原型：GBDT 梯度提升树，本质上是加法模型：训练一堆小的树，不断拟合上一轮模型的残差，所有树的预测结果都加在一起得到最终预测。只用一阶导数（梯度）来优化损失的。
XGBoost：最大升级，用二阶泰勒展开 一阶导数g，二阶导数h
$$L^{(t)} \approx \sum_{i} \left[ g_i w_i + \frac{1}{2} h_i w_i^2 \right] + \Omega(f_t)$$

正则项控制树的复杂度，防止过拟合

- **$L^{(t)}$**：第 $t$ 轮迭代时的目标损失函数。
- **$\approx$**：约等于，这里用了二阶泰勒展开进行近似。
- **$\sum_{i}$**：对所有样本 $i$ 求和。
- **$g_i$**：损失函数在当前预测值处的一阶导数（梯度），即 $g_i = \frac{\partial L}{\partial \hat{y}_i}$。
- **$h_i$**：损失函数在当前预测值处的二阶导数（Hessian，海森值），即 $h_i = \frac{\partial^2 L}{\partial \hat{y}_i^2}$。
- **$w_i$**：第 $i$ 个样本对应的叶子节点的权重（即当前树对样本 $i$ 的预测输出）。
- **$\Omega(f_t)$**：正则化项，控制第 $t$ 棵树的复杂度，防止过拟合。

XGBoost = 牛顿法（二阶泰勒） + 显式正则化 + 稀疏感知 + 特征并行 + 后剪枝/Shrinkage。它把优化算法（二阶导）和工程系统（并行、缓存）做到了极致。

1. 鲁棒性好
2. 正则化：防止过拟合
3. 训练速度快，
4. 支持可输出的

适合的场景：结构化表格数据，量化金融，风控，推荐

图像，文本非结构用深度学习

深度学习的主场为什么是Adam，不是牛顿？
- 存不起
- 噪声
- 鞍点很多
- 非凸
- adam 做随机伪曲率，便宜


牛顿法并没有失败，而是问题的尺度换了。

- 作业1：牛顿法为什么常常显得快？ 牛顿法利用二阶导数捕捉了函数的曲率，通过海森矩阵的逆重构空间（把倾斜峡谷变成标准圆碗），使其具有二阶收敛特性，能不走弯路直奔极小值。
- 作业2：L-BFGS和Adam怎么选？ 数据量小、要求高精度且能全批量（Full-batch）加载时选 L-BFGS；参数量巨大、数据需小批量（Mini-batch）输入且带噪声（如大模型训练）时选 Adam。
- 作业3：什么是收敛半径？ 收敛半径是泰勒级数能精准逼近原函数的最大有效距离范围，超越这个范围级数就会发散失效。
- 作业4：高斯消元是什么？应用？ 高斯消元是通过“前向消元变阶梯、回代倒推求未知数”来解线性方程组的标准化算法，是物理仿真、电路分析及矩阵求逆等科学计算的核心底层工具。